# Abbe Refractometer Example

This notebook demonstrates a ray-tracing simulation of an Abbe refractometer setup
using the `ray_tracing_shapely` library.

> Originally created as a solve.it online notebook and converted to Jupyter format.

### Imports

In [1]:
from IPython.display import display, HTML, clear_output
import os
from typing import List, Dict, Any
import math
import csv
from lxml import etree
from shapely.geometry import LineString
from shapely.geometry.point import Point as ShapPoint
from dataclasses import dataclass, field
from uuid import uuid4

In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
# we use this verbose style to keep track what we are effectively using in the session
# core functions
from ray_tracing_shapely.core.scene import Scene
from ray_tracing_shapely.core.scene_objs.light_source.point_source import PointSource
from ray_tracing_shapely.core.scene_objs.light_source.angle_source import AngleSource
from ray_tracing_shapely.core.scene_objs.glass.ideal_lens import IdealLens
from ray_tracing_shapely.core.scene_objs.glass.spherical_lens import SphericalLens
from ray_tracing_shapely.core.scene_objs.blocker.blocker import Blocker
from ray_tracing_shapely.core.simulator import Simulator
from ray_tracing_shapely.core.ray import Ray
from ray_tracing_shapely.core.svg_renderer import SVGRenderer
from ray_tracing_shapely.core.scene_objs.glass.glass import Glass
from ray_tracing_shapely.core.scene_objs.light_source.single_ray import SingleRay

In [3]:
# analysis functions
from ray_tracing_shapely.analysis import *

## Analysis of examples

## Human playground

in this section I have coded in some examples I want to study more

### example 1: classical Abbe refractometer setup

An Abbe refractometer is a high-precision laboratory instrument used to measure the refractive index of liquids and solids. Its design is ingenious because it allows you to find this index very accurately using just a few drops of a sample.

**Core Principle: The Critical Angle**

- The instrument operates on the principle of the critical angle.
- When light travels from a medium with a lower refractive index (your sample) into one with a higher refractive index (the glass prism), there is a specific maximum angle at which the light can enter.

**Grazing Incidence:** The device shines light through the sample at all possible angles.

- The most important rays are those hitting the prism surface at a "grazing" angle (nearly parallel to the surface).
- The Borderline: These grazing rays are refracted into the prism at the critical angle.
- Because no light can be refracted at an angle larger than this, a sharp "shadow boundary" is created in the field of view.

**Measurement:** You look through an eyepiece and see a circle divided into a bright half and a dark half.
- By turning a knob to align this "shadow line" with a set of crosshairs, you are essentially measuring that critical angle.
- The instrument's scale is pre-calibrated to convert that angle directly into a refractive index ($n_D$) or a sugar concentration (Brix).

**Key Components**

- The Prism Pair: The sample is "sandwiched" between two prisms.
- The bottom illuminating prism has a frosted surface to scatter light in all directions into the sample, while the top measuring prism has a high refractive index to allow for the critical angle effect

Amici Prisms (Compensators): Since white light is made of many colors, it usually creates a blurry, rainbow-colored edge.
- Abbe added two rotating "Amici prisms" that you can adjust to remove this "color fringing," resulting in a sharp black-and-white line.

Water Jacket: Refractive index changes significantly with temperature. Most Abbe refractometers have a metal casing with ports to circulate water from a constant-temperature bath, ensuring the sample stays at a stable temperature (usually 20°C).

---

**1. The Setup: A "Light Sandwich"**

Inside the machine, you put a tiny drop of liquid (your sample) between two glass prisms. The Prism is made of high-index glass (light travels slowly here). The Sample usually has a lower index than the glass (light travels faster here).

**2. Total Internal Reflection (The "Mirror" Effect)**

Normally, we talk about TIR when light tries to go from High Index (Glass) $\rightarrow$ Low Index (Liquid). If the light hits the boundary at a steep enough angle, it can't get out into the liquid. It reflects back into the glass like a mirror. The exact angle where this starts happening is the Critical Angle.

**3. The "Grazing Angle" (TIR in Reverse)**

The Abbe refractometer actually works by looking at the inverse path. Instead of trying to get light out of the glass, it shines light into the glass from the liquid. Grazing Incidence: Imagine a light ray in the liquid traveling almost perfectly flat along the surface of the glass (this is the grazing angle, or $90^\circ$). Because the glass has a higher index, this ray must bend inward as it enters the glass. The Boundary: This "flat" ray enters the glass at exactly the Critical Angle. It is impossible for any light from the liquid to enter the glass at an angle steeper than this.

**4. The Result: A Shadow Line**

Because of that "hard limit" created by the critical angle:
- One side is bright: Light rays hitting the glass at normal angles (like $45^\circ$) enter the glass easily.
- One side is dark: No light can physically exist beyond the critical angle limit.
- The Shadow Line: When you look through the eyepiece, you see a sharp line between light and dark. This line is the critical angle.

| Concept | Direction | Role in the Refractometer |
|---|---|---|
| TIR | Glass $\rightarrow$ Sample | Defines the "Dark" zone where light cannot escape. |
| Grazing Angle | Sample $\rightarrow$ Glass | The "flat" light ray that defines the exact edge of the "Bright" zone. |
| Critical Angle | Boundary | The mathematical "border" that the machine measures to calculate your result. |

#### grazing angle

In [4]:
# vertex coordinates of the measuring prism of Abbe refractometer
grazing_meas_prism_vertex_dict: Dict[str, ShapPoint] = {
    'A': ShapPoint(167, 61),
    'B': ShapPoint(274, 61),
    'C': ShapPoint(188, 104)
}

# vertex coordinates of the medium between prisms of Abbe refractometer
grazing_medium_vertex_dict: Dict[str, ShapPoint] = {
    'A': ShapPoint(167, 61),
    'B': ShapPoint(274, 61),
    'C': ShapPoint(274, 55),
    'D': ShapPoint(167, 55)
}

# vertex coordinates of the optical waveguide
optical_waveguide_dict: Dict[str, ShapPoint] = {
    'D': ShapPoint(167, 55),
    'C': ShapPoint(274, 55),
    'E': ShapPoint(274, 20),
    'F': ShapPoint(167, 20)
}

# these are the settings for the scene.
# Remember that when we increase min_brightness_exp we see more and more the secondary refraction.
# note:
# - for setting up the min_brightness_exp and the grazing_polarization_threshold one can use utilities
#   that calculate Fresnel Tp and Ts values. Below we have fresnel_Tp_Ts_ratio functions.
# - for an angular source angular_step = math.pi * 2 / int(ray_density * 500) so:
#       ray_density     rays/degree
#       0.1             ~0.14
#       1               ~1.4
#       10              ~14
#       72              ~100
#       720             ~1000
grazing_scene_setup_dict: Dict = {
    'min_brightness_exp': 5,  # set 1 to see main optical paths and then 2,3,4, to increasingly track more refraction directions
    'ray_density': 72,
    'color_mode': 'linear',
    'mode': 'rays',
    'show_ray_arrows': True,
    'grazing_polarization_ratio_threshold': 1.4
}

# characteristics of the light source
grazing_light_source_setup_dict: Dict[str, Any] = {
    'emission_angle': 1,
    'symmetric': True,
    'brightness': 1.0,
    'label_pre': 'angle_'
}

In [5]:
@dataclass
class ExampleReturn:
    desc: str
    renderer: SVGRenderer
    scene: Scene
    sim_results: SimulationResult  # this is itself a container with scene, segments (list[Ray])
    # Use default_factory to call uuid4 for every new object
    id: str = field(default_factory=lambda: str(uuid4()))

In [7]:
# code to study the grazing at the interface between the prism and the low refractive medium

def example_grazing_angle(
    meas_prism_ref_index: float = 1.785,  # N-SF11 must be higher than the medium for TIR
    medium_ref_index: float = 1.5,
    wave_guide_ref_index: float = 1.58,
    t: int = 2,  # this is to set the position of the light source in front of the south (S) edge of the medium
    n: int = 10,  # number of divisions of the south edge
    source_rotation_angle: int = 60,  # angle of rotation of the light source in degrees
    source_offset: float = 0.5,  # offset of the light source position
    dic_scene_setup: Dict = grazing_scene_setup_dict,
    meas_prism_vertex_dict: Dict[str, ShapPoint] = grazing_meas_prism_vertex_dict,  # a dictionary where the label is the name of the vertex
    medium_vertex_dict: Dict[str, ShapPoint] = grazing_medium_vertex_dict,  # a dictionary where the label is the name of the vertex
    light_source_setup_dict: Dict[str, Any] = grazing_light_source_setup_dict,
    use_optical_waveguide: bool = False,
    waveguide_vertex_dict: Dict[str, Any] = optical_waveguide_dict,
    verbose: bool = False
) -> ExampleReturn:
    # Create scene and configure grazing thresholds
    scene = Scene()
    scene.ray_density = dic_scene_setup['ray_density']
    scene.color_mode = dic_scene_setup['color_mode']
    scene.mode = dic_scene_setup['mode']
    scene.show_ray_arrows = True
    scene.min_brightness_exp = dic_scene_setup['min_brightness_exp']
    scene.grazing_polarization_ratio_threshold = dic_scene_setup['grazing_polarization_ratio_threshold']

    # Create measuring prism
    meas_prism = Glass(scene)
    meas_prism.path = [
        {'x': meas_prism_vertex_dict['A'].x, 'y': meas_prism_vertex_dict['A'].y, 'arc': False},
        {'x': meas_prism_vertex_dict['B'].x, 'y': meas_prism_vertex_dict['B'].y, 'arc': False},
        {'x': meas_prism_vertex_dict['C'].x, 'y': meas_prism_vertex_dict['C'].y, 'arc': False}
    ]
    meas_prism.not_done = False
    meas_prism.refIndex = meas_prism_ref_index
    meas_prism.name = "measuring prism"

    # Create medium in between prisms
    medium = Glass(scene)
    medium.path = [
        {'x': medium_vertex_dict['A'].x, 'y': medium_vertex_dict['A'].y, 'arc': False},
        {'x': medium_vertex_dict['B'].x, 'y': medium_vertex_dict['B'].y, 'arc': False},
        {'x': medium_vertex_dict['C'].x, 'y': medium_vertex_dict['C'].y, 'arc': False},
        {'x': medium_vertex_dict['D'].x, 'y': medium_vertex_dict['D'].y, 'arc': False}
    ]
    medium.not_done = False
    medium.refIndex = medium_ref_index
    medium.name = "medium"

    if use_optical_waveguide is True:
        # Create optical waveguide
        wave_guide = Glass(scene)
        wave_guide.path = [
            {'x': waveguide_vertex_dict['D'].x, 'y': waveguide_vertex_dict['D'].y, 'arc': False},
            {'x': waveguide_vertex_dict['C'].x, 'y': waveguide_vertex_dict['C'].y, 'arc': False},
            {'x': waveguide_vertex_dict['E'].x, 'y': waveguide_vertex_dict['E'].y, 'arc': False},
            {'x': waveguide_vertex_dict['F'].x, 'y': waveguide_vertex_dict['F'].y, 'arc': False}
        ]
        wave_guide.not_done = False
        wave_guide.refIndex = wave_guide_ref_index
        wave_guide.name = "waveguide"

    scene.add_object(meas_prism)
    scene.add_object(medium)
    if use_optical_waveguide is True:
        scene.add_object(wave_guide)

    # Auto-label all edges with cardinal directions
    scene.auto_label_all_glass_cardinal()

    # Calculate a point in the illumination prism's short side (between points 1 and 2)
    # Point 1: C of medium, Point 2: D of medium
    source_o_point: ShapPoint = ShapPoint()
    source_o_point = interpolate_along_edge(glass_obj=medium, edge_label='S', fraction=t / n, as_point=True)

    # Create angle source
    angle_source = AngleSource(scene)
    if verbose:
        print('source y coord: ', source_o_point.y + source_offset)
    angle_source.p1 = {'x': source_o_point.x, 'y': source_o_point.y + source_offset}  # Source position at middle of short side
    angle_source.p2 = {'x': source_o_point.x, 'y': source_o_point.y + source_offset + 5}  # Reference point (direction towards prism)
    angle_source.emis_angle = light_source_setup_dict['emission_angle']  # Angular spread in degrees
    angle_source.symmetric = light_source_setup_dict['symmetric']  # Symmetric spread around reference direction
    angle_source.brightness = light_source_setup_dict['brightness']

    angle_source.rotate(math.radians(source_rotation_angle))

    # give this light source a label that will percolate to the ray segments
    lsname: str = light_source_setup_dict['label_pre'] + \
        str(light_source_setup_dict['emission_angle']) + '_' + \
        str(source_o_point.x) + '_' + \
        str(source_o_point.y)

    angle_source.name = lsname
    scene.add_object(angle_source)

    # Simulate
    simulator = Simulator(scene)
    sim_res = simulator.run_with_result(name='grazing_example')

    if verbose:
        print(f"Rays traced: {simulator.processed_ray_count}")
        print(f"Segments: {len(sim_res.segments)}")

    # Render
    renderer = SVGRenderer(width=800, height=600, viewbox=(0, 0, 300, 200))
    renderer.draw_scene(scene, sim_res.segments)

    return ExampleReturn('grazing angle', renderer, scene, sim_results=sim_res)

In [8]:
my_res_grazing: ExampleReturn = example_grazing_angle(
    t=1,  # down along the short edge
    source_rotation_angle=89.5,
    source_offset=4.90,
    dic_scene_setup=grazing_scene_setup_dict,
    light_source_setup_dict=grazing_light_source_setup_dict,
    use_optical_waveguide=True,
    wave_guide_ref_index=1.52
)
HTML(my_res_grazing.renderer.to_string())

In [9]:
xml_report = describe_simulation_result(my_res_grazing.sim_results)

In [10]:
# Only rays with extreme polarization ratio
polar_grazing_rays: List[Ray] = filter_grazing_rays(my_res_grazing.sim_results.segments, criterion='polar')

In [11]:
TLDR_print = rays_to_xml(polar_grazing_rays)